# Session 2: Creating APIs with FastAPI

**Learning goals:**

- Retrieve 50 arXiv paper records and save their metadata to a local CSV file.
- Explain how routes, HTTP methods, request models and status codes form an API contract.
- Create FastAPI endpoints that read a CSV file and read or write a JSON file.
- Build a Python client that retrieves papers and saves research notes through HTTP.
- Verify successful requests, missing resources, invalid input and persistent storage.

**Prerequisites:**

- Session 1: HTTP requests and responses, arXiv queries and XML namespaces.
- Python 3.12 or later, a Jupyter kernel, and basic functions, dictionaries and file handling.
- Install `requests`, `pandas`, `fastapi`, `uvicorn` and `ipykernel` in the notebook kernel's environment.
- Internet access for arXiv, and permission to start a local server.
- Open this notebook with the course repository as the working directory.

Work from top to bottom. The three labs share one research catalogue: collect paper metadata, expose local storage through an API, then access it from a client. The examples use only metadata, not downloaded PDF files.


## 1. From consuming APIs to creating APIs

In Session 1, your Python program was the client of arXiv. Here, your own FastAPI application becomes a server. Uvicorn listens for HTTP requests and passes them to FastAPI; FastAPI matches a route, validates input, calls your Python function and serialises the result as JSON.

```text
arXiv API → collection script → local papers.csv
Python client → HTTP → FastAPI route → papers.csv / notes.json
Python client ← JSON response ← FastAPI route ← local storage
```

The client knows the endpoint and JSON contract. The server owns the storage paths. A client on another computer would read the server's files, not files on the client's computer.

### 1.1 Environment and storage

Run this command in a terminal using the same Python environment as your notebook:

```bash
python -m pip install requests pandas fastapi uvicorn ipykernel
```

The collection lab deliberately replaces `sample_data/session2/papers.csv` when rerun. The API writes notes to `sample_data/session2/notes.json`; rerunning a write with the same paper ID replaces that note. These are the only files the API exposes. It accepts paper IDs rather than arbitrary filesystem paths.


## 2. Lab 1: Collect 50 arXiv papers into CSV

### 2.1 Request the Atom feed

Reuse Session 1's query endpoint and namespace. `max_results=50` requests up to 50 entries; it does not guarantee 50 matches. We check the actual count before writing. HTTP errors, timeouts and invalid XML stop the collection rather than producing a misleading CSV.

[Official arXiv API manual](https://info.arxiv.org/help/api/user-manual.html)

Run the request once. If you change the query or retry, leave at least three seconds between arXiv requests.


In [11]:
import requests

ARXIV_API_URL = "https://export.arxiv.org/api/query"
SEARCH_QUERY = 'all:"machine learning"'
PAPER_LIMIT = 50
REQUEST_TIMEOUT_SECONDS = 30

# Pass query options separately so requests encodes them in the URL.
arxiv_response = requests.get(
    ARXIV_API_URL,
    params={
        "search_query": SEARCH_QUERY,
        "start": 0,  # Index of the first result.
        "max_results": PAPER_LIMIT,  # Requested number of entries.
        "sortBy": "submittedDate",  # Order papers by submission date.
        "sortOrder": "descending",  # Most recent entries first.
    },
    timeout=REQUEST_TIMEOUT_SECONDS,  # Bound the wait for a response.
)
arxiv_response.raise_for_status()  # Stop if the HTTP request failed.
print("HTTP status:", arxiv_response.status_code)


HTTP status: 200


### 2.2 Parse paper metadata

Keep the versioned arXiv identifier, title, authors, publication date, summary and source URL. CSV stores text columns; the semicolon-separated authors field remains one column. XML namespaces identify Atom elements. Whitespace is normalised to make titles and summaries easier to read.


In [12]:
import xml.etree.ElementTree as ET

ATOM_NAMESPACE = {"atom": "http://www.w3.org/2005/Atom"}
arxiv_root = ET.fromstring(arxiv_response.content)  # Parse the XML response bytes.
papers = []
for entry in arxiv_root.findall("atom:entry", ATOM_NAMESPACE):
    def atom_text(tag):
        # Use the Atom namespace and return an empty string for missing fields.
        return entry.findtext(f"atom:{tag}", default="", namespaces=ATOM_NAMESPACE).strip()

    source_url = atom_text("id")
    if "/abs/" not in source_url:
        raise ValueError("Expected an arXiv paper entry; inspect the API response.")
    papers.append({
        "paper_id": source_url.split("/abs/", 1)[1],
        "title": " ".join(atom_text("title").split()),
        "authors": "; ".join(
            author.findtext("atom:name", default="", namespaces=ATOM_NAMESPACE)
            for author in entry.findall("atom:author", ATOM_NAMESPACE)
        ),
        "published": atom_text("published"),
        "summary": " ".join(atom_text("summary").split()),
        "source_url": source_url,
    })

if len(papers) != PAPER_LIMIT:
    raise ValueError(f"Expected {PAPER_LIMIT} papers, received {len(papers)}. Check the query.")
if len({paper["paper_id"] for paper in papers}) != PAPER_LIMIT:
    raise ValueError("Duplicate paper IDs found.")
print("Collected papers:", len(papers))
print("First title:", papers[0]["title"])


Collected papers: 50
First title: Hide and Seek: HIDE Concepts to SEEK explanations for black box models


### 2.3 Save and reopen the CSV with pandas

A pandas `DataFrame` organises the paper dictionaries into rows and columns. `to_csv()` writes the table; `index=False` excludes the DataFrame's row index from the file. UTF-8 preserves non-ASCII titles and author names.

Reopen the file with `read_csv()` to check the stored row count. `dtype=str` preserves identifiers and dates as text. `keep_default_na=False` keeps empty fields as empty strings rather than missing values (`NaN`). Both methods handle quoted commas and embedded newlines in CSV fields.


In [13]:
import pandas as pd
import os

DATA_DIRECTORY = "sample_data/session2"
PAPERS_CSV = f"{DATA_DIRECTORY}/papers.csv"
CSV_FIELDS = ["paper_id", "title", "authors", "published", "summary", "source_url"]
os.makedirs(DATA_DIRECTORY, exist_ok=True)  # Create the data directory if needed.
papers_table = pd.DataFrame(papers, columns=CSV_FIELDS)  # Set the table's column order.
papers_table.to_csv(
    PAPERS_CSV,
    index=False,  # Save only the data columns, without the DataFrame row index.
    encoding="utf-8",  # Preserve Unicode titles and author names.
)

saved_papers = pd.read_csv(
    PAPERS_CSV,
    dtype=str,  # Keep identifiers and dates as text.
    keep_default_na=False,  # Preserve empty fields as empty strings.
    encoding="utf-8",
)
assert len(saved_papers) == PAPER_LIMIT
assert saved_papers.columns.tolist() == CSV_FIELDS
print(f"Verified {len(saved_papers)} rows in {PAPERS_CSV}")
print(saved_papers[["paper_id", "title"]].head())  # Preview the first five stored records.


Verified 50 rows in sample_data/session2/papers.csv
       paper_id                                              title
0  2610.13935v1  Hide and Seek: HIDE Concepts to SEEK explanati...
1  2610.13931v1  Constructive Bounds for the Topological Expres...
2  2610.13928v1  AI Models Can Provably Hide Arbitrary Capabili...
3  2610.13924v1  KL Before and After Mixing: A Round-Complexity...
4  2610.13923v1  Goal-Oriented Bundle Management and Flow Contr...


## 3. Lab 2: Create a FastAPI service for local files

### 3.1 Design the API contract

The catalogue is read-only through this API. A separate notes resource demonstrates file writing without replacing the collected papers. `PUT` creates or replaces the note for one existing paper and returns `200` with the stored representation. Repeating the same `PUT` produces the same stored note.

| Method | Endpoint | Purpose | Success |
|---|---|---|---|
| GET | `/health` | Check that this service is running | 200 |
| GET | `/papers?limit=5` | Read CSV records with a bounded result count | 200 |
| GET | `/notes?paper_id=...` | Read one saved note from JSON | 200 |
| PUT | `/notes` | Create or replace a note in JSON | 200 |

An unknown paper or missing note returns `404`. Invalid input returns `422`. A missing CSV returns `503`, explaining that Lab 1 must run first. Query parameters support both modern arXiv IDs and older IDs containing `/`.

[FastAPI first steps](https://fastapi.tiangolo.com/tutorial/first-steps/) · [Request bodies and Pydantic models](https://fastapi.tiangolo.com/tutorial/body/)

### 3.2 Application and request model

The route decorator associates a function with a method and URL. `BaseModel` defines the incoming JSON shape; `Field` constrains text lengths. `Query` constrains the optional result limit. Synchronous `def` handlers suit this small example's blocking file operations.


In [14]:
import json
import threading
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

app = FastAPI(title="Research Catalogue API")  # Used in generated API documentation.
NOTES_JSON = f"{DATA_DIRECTORY}/notes.json"
storage_lock = threading.Lock()  # Serialise file access within this server process.

class NoteInput(BaseModel):
    paper_id: str = Field(min_length=1, max_length=100)  # Required catalogue identifier.
    note: str = Field(min_length=1, max_length=2000)  # Bound the note size.

@app.get("/health")  # Register a GET route.
def health():
    return {"status": "ok", "service": "research-catalogue"}


### 3.3 Read the CSV on the server

Each request loads the CSV into a DataFrame with `pd.read_csv()`, so changes made by rerunning Lab 1 appear on the next request. `to_dict(orient="records")` converts each row to a dictionary for the JSON response. The server decides which file to open.


In [15]:
def read_papers():
    try:
        catalogue = pd.read_csv(
            PAPERS_CSV,
            dtype=str,  # Keep identifiers and dates as text.
            keep_default_na=False,  # Keep empty fields as strings rather than NaN.
            encoding="utf-8",  # Read Unicode titles and author names.
        )
        return catalogue.to_dict(orient="records")  # Convert rows to JSON-ready dictionaries.
    except FileNotFoundError:
        raise HTTPException(status_code=503, detail="Run Lab 1 to create papers.csv.")

@app.get("/papers")
def list_papers(limit: int = Query(default=5, ge=1, le=50)):
    # Accept a query parameter between 1 and 50.
    catalogue = read_papers()
    return {"total": len(catalogue), "papers": catalogue[:limit]}


### 3.4 Read and write notes in JSON

The JSON file maps paper IDs to notes. An absent notes file represents an empty collection. For updates, read the current collection, change one entry, write a temporary file and atomically replace the destination. Other saved notes are retained. The lock protects this read-modify-write sequence in one process; this lesson uses one server worker. A database is more appropriate for multiple workers and larger applications.


In [16]:
def read_notes():
    try:
        with open(NOTES_JSON, "r", encoding="utf-8") as notes_file:
            return json.load(notes_file)  # Convert JSON text to a dictionary.
    except FileNotFoundError:
        return {}

@app.get("/notes")
def get_note(paper_id: str = Query(min_length=1, max_length=100)):
    with storage_lock:
        notes = read_notes()
    if paper_id not in notes:
        raise HTTPException(status_code=404, detail="No note saved for this paper.")
    return {"paper_id": paper_id, "note": notes[paper_id]}

@app.put("/notes")
def save_note(payload: NoteInput):
    if not payload.note.strip():
        raise HTTPException(status_code=422, detail="The note must contain text.")
    with storage_lock:
        if payload.paper_id not in {paper["paper_id"] for paper in read_papers()}:
            raise HTTPException(status_code=404, detail="Paper not found in the catalogue.")
        notes = read_notes()
        notes[payload.paper_id] = payload.note
        temporary_file = f"{NOTES_JSON}.tmp"
        with open(temporary_file, "w", encoding="utf-8") as notes_file:
            json.dump(notes, notes_file, ensure_ascii=False, indent=2)  # Preserve Unicode.
        os.replace(temporary_file, NOTES_JSON)  # Replace the file after writing completes.
    return payload.model_dump()  # Return the validated stored representation.


### 3.5 Start the HTTP server

The notebook starts Uvicorn in a background thread so subsequent client cells can run. The operating system selects a free port; use the printed URL to visit `/docs` and try the endpoints in Swagger UI. The server binds to `127.0.0.1`, so this unauthenticated classroom example is available only on your own computer. Run the final shutdown cell before rebuilding the app or closing the kernel.


In [17]:
import socket
import time
import uvicorn

server_socket = socket.socket()
server_socket.bind(("127.0.0.1", 0))  # Port 0 asks the operating system for a free port.
SERVER_PORT = server_socket.getsockname()[1]
API_BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"
server = uvicorn.Server(uvicorn.Config(app, log_level="warning"))
server_thread = threading.Thread(
    target=server.run,
    kwargs={"sockets": [server_socket]},  # Serve HTTP using the reserved socket.
    daemon=True,
)
server_thread.start()
try:
    for attempt in range(100):
        if server.started:
            break
        if not server_thread.is_alive():
            raise RuntimeError("The server stopped during startup.")
        time.sleep(0.1)  # Briefly wait before checking readiness again.
    else:
        raise RuntimeError("The server did not start within 10 seconds.")
except Exception:
    server.should_exit = True
    server_thread.join(timeout=5)
    server_socket.close()
    raise
print("API:", API_BASE_URL)
print("Interactive documentation:", f"{API_BASE_URL}/docs")


API: http://127.0.0.1:55663
Interactive documentation: http://127.0.0.1:55663/docs


## 4. Lab 3: Create a Python client application

### 4.1 Retrieve papers through HTTP

The client uses `requests` and has no access to the server's storage functions. A timeout bounds each network request; `raise_for_status()` stops unsuccessful operations before parsing JSON. `params` encodes query parameters and `json` sends a JSON request body with the appropriate content type.


In [18]:
def api_request(method, endpoint, **options):
    response = requests.request(
        method,
        f"{API_BASE_URL}{endpoint}",
        timeout=REQUEST_TIMEOUT_SECONDS,  # Bound the wait for the local API.
        **options,
    )
    response.raise_for_status()  # Reject unsuccessful responses before JSON parsing.
    return response.json()

catalogue_response = api_request("GET", "/papers", params={"limit": 5})
print("Total papers:", catalogue_response["total"])
for paper in catalogue_response["papers"]:
    print(paper["paper_id"], "—", paper["title"])


Total papers: 50
2610.13935v1 — Hide and Seek: HIDE Concepts to SEEK explanations for black box models
2610.13931v1 — Constructive Bounds for the Topological Expressivity of Neural Networks
2610.13928v1 — AI Models Can Provably Hide Arbitrary Capabilities
2610.13924v1 — KL Before and After Mixing: A Round-Complexity Separation in Parallel Sampling
2610.13923v1 — Goal-Oriented Bundle Management and Flow Control for Deep-Space Communications via Reinforcement Learning


### 4.2 Write a note and read it back

Select an ID returned by the server. The client writes through `PUT /notes` and confirms the stored value through a separate `GET /notes` request. No client-side `open()` call is needed.


In [19]:
selected_paper_id = catalogue_response["papers"][0]["paper_id"]
note_payload = {
    "paper_id": selected_paper_id,
    "note": "Review this paper's methods and dataset before the next class.",
}
stored_note = api_request("PUT", "/notes", json=note_payload)  # Send the JSON body.
retrieved_note = api_request("GET", "/notes", params={"paper_id": selected_paper_id})
assert retrieved_note == stored_note == note_payload
print("Saved and retrieved:", retrieved_note)


Saved and retrieved: {'paper_id': '2610.13935v1', 'note': "Review this paper's methods and dataset before the next class."}


### 4.3 Check rejected requests

For deliberately invalid requests, inspect status codes directly instead of calling `raise_for_status()`. These checks show that input validation occurs before storage changes and that unknown IDs are rejected.


In [20]:
invalid_note = requests.put(
    f"{API_BASE_URL}/notes",
    json={"paper_id": selected_paper_id, "note": ""},  # Empty text violates min_length.
    timeout=REQUEST_TIMEOUT_SECONDS,
)
unknown_paper = requests.put(
    f"{API_BASE_URL}/notes",
    json={"paper_id": "not-in-the-catalogue", "note": "Test note"},
    timeout=REQUEST_TIMEOUT_SECONDS,
)
invalid_limit = requests.get(
    f"{API_BASE_URL}/papers", params={"limit": 0}, timeout=REQUEST_TIMEOUT_SECONDS,
)
assert invalid_note.status_code == 422
assert unknown_paper.status_code == 404
assert invalid_limit.status_code == 422
assert api_request("GET", "/notes", params={"paper_id": selected_paper_id}) == note_payload
print("Empty note:", invalid_note.status_code)
print("Unknown paper:", unknown_paper.status_code)
print("Invalid limit:", invalid_limit.status_code)


Empty note: 422
Unknown paper: 404
Invalid limit: 422


### 4.4 Run the standalone applications

The accompanying [server](assets/session2/server.py) and [client](assets/session2/client.py) use the same API contract. After Lab 1, open two terminals in the course repository and activate the environment containing the installed packages.

In terminal 1, start a separate server on port 8000:

```bash
python assets/session2/server.py
```

In terminal 2, run the interactive client:

```bash
python assets/session2/client.py
```

Choose **list**, copy a paper ID, choose **write**, then choose **read** to retrieve the note. The client displays connection errors and API validation messages. Type **quit** to close it. Stop the terminal server with Ctrl+C. If port 8000 is already in use, stop the existing application before starting this server.



## 5. Stop the notebook server and troubleshoot

Run this cell when finished. If a client cell fails, you can still run it to release the server socket. To run the demonstration again, restart the kernel and run the notebook from the beginning.


In [21]:
server.should_exit = True  # Ask Uvicorn to shut down gracefully.
server_thread.join(timeout=5)  # Wait for the background server thread to finish.
server_socket.close()
print("Server stopped:", not server_thread.is_alive())


Server stopped: True


| Symptom | Check or action |
|---|---|
| arXiv timeout, `429` or `503` | Wait and retry the collection later; do not substitute invented records. |
| Fewer than 50 results | Broaden the search query and request 50 again. |
| `ModuleNotFoundError` | Install packages in the environment used by the selected kernel. |
| API `503` about the CSV | Complete Lab 1 and check the repository working directory. |
| Client connection error | Start the server and use its actual base URL and port. |
| API `404` for a note | Save a note first and use the same paper ID. |
| API `422` | Inspect the error detail and compare your input with the request model. |
| Invalid JSON in notes storage | Stop the server and inspect the JSON file; preserve a copy before repairing it. |

Discuss: which parts would need to change if the storage moved from files to a database? Which parts of the client could stay the same?
